# 🎓 Entrenamiento Definitivo Qwen 2.5 - Acervo Total Biblioteca Central UABC
### Proyecto MTAO - FCITEC UABC | Autor: L.S.C. Ana Gabriela Molina Moreno / Desarrollador: Adrián

Este notebook realiza el **Fine-Tuning conjunto (LoRA)** sobre **10,000 ejemplos reales** de todo el catálogo oficial de la Biblioteca Central Mexicali en una sola sesión de GPU T4 (~22 a 25 min).

**¿Por qué entrenar las 4 partes juntas?**
Para evitar el *olvido catastrófico* (que la IA olvide ingeniería al aprender derecho) y porque los archivos `.gguf` son compilados finales que no pueden reentrenarse. Entrenar todo junto genera el **modelo maestro definitivo** para toda la universidad:
- 💻 **Ingeniería y Tecnología** (Sistemas, IA, Robótica, Telecomunicaciones, Civil, Mecánica, Álgebra)
- ⚖️ **Ciencias Sociales y Derecho** (Constitucional, Amparo, Historia UABC/BC, Literatura, Psicología)
- 📊 **Administración y Economía** (Finanzas, NIF, Auditoría, Marketing, Recursos Humanos, Negocios)
- 🩺 **Salud y Ciencias Básicas** (Medicina, Fisiología, Farmacología, Enfermería, Física, Química, Arquitectura)

## Paso 1: Instalar dependencias optimizadas (Unsloth)

In [ ]:
# Instalación de Unsloth y aceleradores
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## Paso 2: Cargar el Modelo Base Qwen 2.5 en 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None # Auto detección Float16 en T4
load_in_4bit = True # 4 bits para mínimo consumo de memoria

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## Paso 3: Configurar Adaptadores LoRA Especializados (r=32, alpha=64)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## Paso 4: Cargar el Dataset Maestro de la Biblioteca Central
> Sube el archivo `dataset_koha_completo_10000_ejemplos.jsonl` a la pestaña de archivos (📁) de Colab.
> *(O si subiste `dataset_koha_3partes_completas.jsonl`, también lo detecta automáticamente)*.

In [ ]:
import os
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convs]
    return { "text" : texts }

# Detecta automáticamente el archivo subido
if os.path.exists("dataset_koha_completo_10000_ejemplos.jsonl"):
    data_file = "dataset_koha_completo_10000_ejemplos.jsonl"
elif os.path.exists("dataset_koha_3partes_completas.jsonl"):
    data_file = "dataset_koha_3partes_completas.jsonl"
else:
    data_file = "dataset_koha_parte1_ingenieria.jsonl"

print(f"Cargando dataset: {data_file}...")
dataset = load_dataset("json", data_files=data_file, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"¡Dataset cargado! Muestras totales listas para entrenar: {len(dataset)}")


## Paso 5: Entrenar el Modelo Maestro (~20-24 min en GPU T4)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Calculo dinámico de pasos según el tamaño del dataset (~2 épocas completas)
total_samples = len(dataset)
effective_batch = 16 # 4 * 4
target_steps = min(1200, int((total_samples / effective_batch) * 2))
print(f"Ejecutando entrenamiento calibrado: {target_steps} pasos para {total_samples} ejemplos...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4, # Batch efectivo = 16
        warmup_steps = 30,
        max_steps = target_steps,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 25,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("\n¡Entrenamiento completado exitosamente!")


## Paso 6: Prueba en Vivo Multidisciplinaria (Ingeniería, Derecho, Finanzas, Salud y Horarios)

In [ ]:
FastLanguageModel.for_inference(model)

preguntas_prueba = [
    "paro we, ocupo el libro de robótica de Olivares",
    "¿Qué libros tienen sobre derecho constitucional mexicano y juicio de amparo?",
    "buenas tardes, busco bibliografía de contabilidad de costos y NIF",
    "¿Tienen en sala el libro de fisiología médica de Guyton?",
    "¿A qué hora cierran la biblioteca central hoy?"
]

for p in preguntas_prueba:
    messages = [
        {"role": "system", "content": "Eres el asistente inteligente de la Biblioteca Central Campus Mexicali (UABC). Tu función es analizar la consulta del usuario en lenguaje natural y extraer la intención y parámetros de búsqueda para la API de KOHA en formato JSON."},
        {"role": "user", "content": p}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize = True, add_generation_prompt = True, return_tensors = "pt").to("cuda")
    outputs = model.generate(input_ids = inputs, max_new_tokens = 100, use_cache = True)
    resp = tokenizer.batch_decode(outputs)[0]
    print(f"Pregunta: {p}")
    print("JSON:", resp.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip())
    print("-"*60)


## Paso 7: Exportar el Modelo Maestro a GGUF y Descargar

In [ ]:
# Exportar a GGUF cuantizado (q4_k_m)
model.save_pretrained_gguf("qwen_koha_uabc_model", tokenizer, quantization_method = "q4_k_m")
print("¡Modelo GGUF exportado con éxito en qwen_koha_uabc_model_gguf!")

# Descarga directa en el navegador
from google.colab import files
files.download("qwen_koha_uabc_model_gguf/Qwen2.5-0.5B-Instruct.Q4_K_M.gguf")
